# Synthetic Data Generation for RAG Evaluation

Session 1 built a vector RAG application over a cat health guideline PDF. This
session creates an evaluation dataset for that application and uses the dataset
to compare two retrieval configurations. All generation, embedding, RAG, and
judge requests are routed through Vercel AI Gateway.

The workflow is:

~~~text
corpus -> knowledge graph -> synthetic examples -> human review
       -> LangSmith dataset -> baseline and candidate experiments
~~~

Synthetic examples reduce the cost of getting started, but generated references
are not automatically ground truth. We will inspect and curate them before using
them as evaluation targets.

> This is an educational cat health exercise, not veterinary advice. Generated
> questions and answers must not be used to diagnose, prescribe, or replace a
> veterinarian.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain how Ragas builds a knowledge graph for test data generation.
- Distinguish single-hop specific, multi-hop specific, and multi-hop abstract queries.
- Generate and review synthetic questions, reference contexts, and reference answers.
- Route generation, embeddings, RAG, and judge calls through Vercel AI Gateway.
- Upload reviewed examples to a LangSmith dataset.
- Evaluate answer correctness, answer groundedness, and retrieval relevance.
- Run a controlled RAG experiment that changes one variable at a time.

## Table of Contents

- **Breakout Room #1: Synthetic Test Data with Ragas**
  - Task 1: Environment Setup
  - Task 2: Load the Cat Health Corpus
  - Task 3: Build and Enrich a Knowledge Graph
  - Task 4: Inspect the Query Distribution
  - Task 5: Generate and Inspect a Synthetic Test Set
  - Activity #1: Review and Curate the Dataset
- **Breakout Room #2: RAG Evaluation with LangSmith**
  - Task 6: Create a LangSmith Dataset
  - Task 7: Build a Baseline RAG Application
  - Task 8: Define RAG Evaluators
  - Task 9: Run the Baseline Experiment
  - Task 10: Change One Retrieval Variable and Re-Evaluate
  - Activity #2: Compare, Diagnose, and Iterate
  - Advanced Build: Add Robustness and Adversarial Cases

---
# Breakout Room #1
## Synthetic Test Data with Ragas

Ragas uses the source corpus to create a richer representation of its topics and
relationships. Query synthesizers then select scenarios from that representation
and generate questions plus reference answers.

The knowledge graph is a generation aid. It is not the graph used by the RAG
application in Breakout Room #2.

## Task 1: Environment Setup

From the <code>05_Synthetic_Data_Generation_for_RAG_Evals</code> folder:

~~~bash
uv sync
~~~

Then select the environment created by uv as this notebook's kernel.

Required accounts:

- Vercel AI Gateway for generation, embeddings, the RAG answer model, and judges
- LangSmith for the dataset and experiments

A direct OpenAI API key is not required. The OpenAI SDK is used only as a
protocol-compatible client pointed at Vercel AI Gateway.

The default synthetic test set is intentionally small. Ragas generation and
LLM-as-judge evaluation both make multiple model calls, so start small and scale
only after inspecting quality.

### Imports

In [1]:
from __future__ import annotations

import os
from collections import Counter
from getpass import getpass
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

import instructor
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel, field_validator

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import Client, evaluate
from openevals.llm import create_llm_as_judge
from openevals.prompts import (
    CORRECTNESS_PROMPT,
    RAG_GROUNDEDNESS_PROMPT,
    RAG_RETRIEVAL_RELEVANCE_PROMPT,
)

from ragas.embeddings import embedding_factory
from ragas.llms import llm_factory
from ragas.run_config import RunConfig
from ragas.testset import TestsetGenerator
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.synthesizers import (
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
    SingleHopSpecificQuerySynthesizer,
    default_query_distribution,
)
from ragas.testset.transforms import (
    CustomNodeFilter,
    SummaryExtractor,
    apply_transforms,
    default_transforms_for_prechunked,
)

c:\Git\The-AI-Engineering-Certification-v1.0\05_Synthetic_Data_Generation_for_RAG_Evals\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### API Keys, Models, and Cost Controls

The notebook reads model names and budgets from environment variables so you can
tune cost without editing every cell. Vercel AI Gateway exposes an
OpenAI-compatible endpoint, so the OpenAI and LangChain clients only need a
different API key, base URL, and provider-qualified model ID.

See the [Vercel AI Gateway Python documentation](https://vercel.com/docs/ai-gateway/sdks-and-apis/python)
for the current authentication and endpoint details.

LangSmith uses <code>LANGSMITH_TRACING</code>. The older
<code>LANGCHAIN_TRACING_V2</code> name from the source notebook is no longer
needed here.

In [2]:
gateway_api_key = (
    os.environ.get("AI_GATEWAY_API_KEY")
    or os.environ.get("VERCEL_OIDC_TOKEN")
)

if not gateway_api_key:
    gateway_api_key = getpass("Vercel AI Gateway API Key: ")
    os.environ["AI_GATEWAY_API_KEY"] = gateway_api_key

if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass("LangSmith API Key: ")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ.setdefault(
    "LANGSMITH_PROJECT",
    "aim-session-5-synthetic-rag-evals",
)

GATEWAY_BASE_URL = os.environ.get(
    "AI_GATEWAY_BASE_URL",
    "https://ai-gateway.vercel.sh/v1",
)
GENERATOR_MODEL_NAME = os.environ.get(
    "AIM_GENERATOR_MODEL",
    "openai/gpt-5.4-mini",
)
RAG_MODEL_NAME = os.environ.get(
    "AIM_RAG_MODEL",
    "openai/gpt-5.4-mini",
)
JUDGE_MODEL_NAME = os.environ.get(
    "AIM_JUDGE_MODEL",
    "openai/gpt-5.4-mini",
)
EMBEDDING_MODEL_NAME = os.environ.get(
    "AIM_EMBEDDING_MODEL",
    "openai/text-embedding-3-small",
)
TESTSET_SIZE = int(os.environ.get("AIM_TESTSET_SIZE", "6"))
MAX_CONCURRENCY = int(os.environ.get("AIM_MAX_CONCURRENCY", "2"))

gateway_models = {
    "generator": GENERATOR_MODEL_NAME,
    "rag": RAG_MODEL_NAME,
    "judge": JUDGE_MODEL_NAME,
    "embedding": EMBEDDING_MODEL_NAME,
}
for role, model_name in gateway_models.items():
    if "/" not in model_name:
        raise ValueError(
            f"{role} model must use a provider-qualified AI Gateway ID: "
            f"{model_name!r}"
        )

print(f"Ragas: {version('ragas')}")
print(f"LangSmith: {version('langsmith')}")
print(f"AI Gateway: {GATEWAY_BASE_URL}")
print(f"Generator model: {GENERATOR_MODEL_NAME}")
print(f"RAG model: {RAG_MODEL_NAME}")
print(f"Judge model: {JUDGE_MODEL_NAME}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Synthetic examples: {TESTSET_SIZE}")
print(f"LangSmith tracing: {os.environ['LANGSMITH_TRACING']}")

Ragas: 0.4.4.dev8+g298b68274
LangSmith: 0.8.18
AI Gateway: https://ai-gateway.vercel.sh/v1
Generator model: openai/gpt-5.4-mini
RAG model: openai/gpt-5.4-mini
Judge model: openai/gpt-5.4-mini
Embedding model: openai/text-embedding-3-small
Synthetic examples: 6
LangSmith tracing: true


## Task 2: Load the Cat Health Corpus

The corpus is the bundled 2021 AAHA/AAFP Feline Life Stage Guidelines PDF used
in Session 1. <code>PyPDFLoader</code> extracts one LangChain document per page,
including page metadata that survives later chunking.

This gives the generator multiple related units to connect:

- hydration and urinary signs
- preventive care and senior care
- dental pain and behavior changes
- monitoring and emergency escalation

In [3]:
corpus_path = Path("data/cat_health_guidelines.pdf")

if not corpus_path.exists():
    raise FileNotFoundError(
        f"Expected the course corpus at {corpus_path.resolve()}"
    )

pdf_loader = PyPDFLoader(str(corpus_path))
source_documents = pdf_loader.load()
source_documents = [
    document
    for document in source_documents
    if len(document.page_content.strip()) >= 200
]

for index, document in enumerate(source_documents):
    page_number = int(document.metadata.get("page", index)) + 1
    document.metadata.update(
        {
            "source": corpus_path.name,
            "document_type": "feline_life_stage_guidelines",
            "page_number": page_number,
        }
    )

print(f"Loaded {len(source_documents)} text-containing PDF pages")
for document in source_documents[:5]:
    page_number = document.metadata["page_number"]
    print(
        f"- page {page_number}: "
        f"{len(document.page_content)} characters"
    )

Loaded 20 text-containing PDF pages
- page 1: 4913 characters
- page 2: 2084 characters
- page 3: 5955 characters
- page 6: 5673 characters
- page 7: 3588 characters


Inspect one PDF page and its metadata. The metadata is useful for debugging,
trace inspection, and explaining where a retrieved chunk came from.

In [4]:
sample_document = source_documents[0]

print(sample_document.metadata)
print()
print(sample_document.page_content[:800])

{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1', 'document_type': 'feline_life_stage_guidelines', 'page_number': 1}

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177

## Task 3: Build and Enrich a Knowledge Graph

The unrolled workflow makes the generation stages visible:

1. Treat each text-containing PDF page as a pre-chunked Ragas node.
2. Run Ragas extractors, embeddings, and relationship builders.
3. Save the graph so expensive enrichment can be reused.

Ragas remains responsible for graph enrichment and synthetic generation. The
newer pinned Ragas build exposes an official Instructor mode parameter, so its
LLM factory can use AI Gateway tool calls directly without custom wrappers.

In [5]:
gateway_client = OpenAI(
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

generator_llm = llm_factory(
    GENERATOR_MODEL_NAME,
    provider="openai",
    client=gateway_client,
    mode=instructor.Mode.TOOLS,
    max_tokens=4096,
)
# Provider-qualified Gateway IDs bypass Ragas's GPT-5 parameter detection.
# Keep only the token limit supported by the Gateway route. max_retries is
# consumed locally by Instructor and is not sent to AI Gateway.
generator_llm.model_args = {
    "max_tokens": 4096,
    "max_retries": 3,
}

generator_embeddings = embedding_factory(
    "openai",
    model=EMBEDDING_MODEL_NAME,
    client=gateway_client,
)

ragas_run_config = RunConfig(
    timeout=180,
    max_retries=3,
    max_wait=30,
    max_workers=MAX_CONCURRENCY,
)

C:\Users\dawel\AppData\Local\Temp\ipykernel_28632\1199448542.py:21: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = embedding_factory(


Before building the graph, make one small structured-output request through
Ragas. This catches gateway authentication, model availability, and tool-calling
incompatibilities without waiting for every PDF page to retry.

In [6]:
class GatewayToolCallCheck(BaseModel):
    status: str


class NonEmptySummary(BaseModel):
    text: str

    @field_validator("text")
    @classmethod
    def require_text(cls, value):
        value = value.strip()
        if not value:
            raise ValueError("summary text cannot be empty")
        return value


gateway_check = generator_llm.generate(
    "Use the required tool with a short, non-empty status message.",
    GatewayToolCallCheck,
)
if not gateway_check.status.strip():
    raise RuntimeError("AI Gateway returned an empty tool-call check")

print(f"AI Gateway tool-based structured output: {gateway_check.status}")

AI Gateway tool-based structured output: ok


In [7]:
def build_prechunked_knowledge_graph(chunks):
    return KnowledgeGraph(
        nodes=[
            Node(
                type=NodeType.CHUNK,
                properties={
                    "page_content": chunk.page_content,
                    "document_metadata": dict(chunk.metadata),
                },
            )
            for chunk in chunks
            if chunk.page_content.strip()
        ]
    )


generation_chunks = list(source_documents)
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)

print(f"Ragas input chunks: {len(generation_chunks)}")
print(knowledge_graph)

Ragas input chunks: 20
KnowledgeGraph(nodes: 20, relationships: 0)


### Apply Ragas Transforms

Because the PDF loader already gives us coherent page-level chunks, use Ragas'
built-in pre-chunked transform pipeline. It skips headline extraction and
splitting, then applies Ragas summaries, embeddings, themes, named entities,
and relationship builders directly to the PDF pages. The parent-child node
filter is omitted because these page chunks intentionally have no parent nodes.
A non-empty output constraint makes Instructor retry blank Ragas summaries before
the built-in embedding transform runs.

In [8]:
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)
transforms = [
    transform
    for transform in default_transforms_for_prechunked(
        llm=generator_llm,
        embedding_model=generator_embeddings,
    )
    if not isinstance(transform, CustomNodeFilter)
]

summary_transform = next(
    transform
    for transform in transforms
    if isinstance(transform, SummaryExtractor)
)
summary_transform.prompt.output_model = NonEmptySummary

print("Ragas transform pipeline:")
for transform in transforms:
    nested = getattr(transform, "transformations", None)
    if nested is None:
        print(f"- {type(transform).__name__}")
    else:
        names = ", ".join(type(item).__name__ for item in nested)
        print(f"- Parallel({names})")

for transform in transforms:
    apply_transforms(
        knowledge_graph,
        transform,
        run_config=ragas_run_config,
    )
    if isinstance(transform, SummaryExtractor):
        empty_summary_nodes = [
            node
            for node in knowledge_graph.nodes
            if not str(node.get_property("summary") or "").strip()
        ]
        if empty_summary_nodes:
            raise RuntimeError(
                "Ragas did not produce non-empty summaries for "
                f"{len(empty_summary_nodes)} PDF chunks"
            )

print(knowledge_graph)

Ragas transform pipeline:
- SummaryExtractor
- Parallel(EmbeddingExtractor, ThemesExtractor, NERExtractor)
- Parallel(CosineSimilarityBuilder, OverlapScoreBuilder)


Applying SummaryExtractor: 100%|██████████| 20/20 [00:41<00:00,  2.07s/it]
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/60 [00:00<?, ?it/s]c:\Git\The-AI-Engineering-Certification-v1.0\05_Synthetic_Data_Generation_for_RAG_Evals\.venv\Lib\site-packages\ragas\testset\transforms\base.py:198: UserWarning: Using sync embedding model OpenAIEmbeddings in async context. This may impact performance. Consider using an async-compatible embedding model for better performance.
  property_name, property_value = await self.extract(node)
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]: 100%|██████████| 60/60 [01:09<00:00,  1.16s/it]
Applying [CosineSimilarityBuilder, OverlapScoreBuilder]: 100%|██████████| 2/2 [00:00<00:00, 111.89it/s]

KnowledgeGraph(nodes: 20, relationships: 43)


Inspect the graph at a high level. Different Ragas versions may add different
properties, so the notebook avoids depending on one exact internal schema.

In [9]:
node_type_counts = Counter(str(node.type) for node in knowledge_graph.nodes)

print("Node types:")
for node_type, count in node_type_counts.items():
    print(f"- {node_type}: {count}")

print(f"Relationships: {len(knowledge_graph.relationships)}")

for index, node in enumerate(knowledge_graph.nodes[:3], start=1):
    property_names = sorted(node.properties.keys())
    print(f"Node {index} properties: {property_names}")

Node types:
- NodeType.CHUNK: 20
Relationships: 43
Node 1 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 2 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 3 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']


### Save and Reload the Graph

Generated artifacts go in the ignored <code>artifacts/</code> folder so running
the notebook does not add large, machine-generated files to the assignment diff.

In [10]:
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

knowledge_graph_path = artifacts_dir / "cat_health_knowledge_graph.json"
knowledge_graph.save(str(knowledge_graph_path))

loaded_knowledge_graph = KnowledgeGraph.load(str(knowledge_graph_path))

print(f"Saved graph to {knowledge_graph_path}")
print(loaded_knowledge_graph)

Saved graph to artifacts\cat_health_knowledge_graph.json
KnowledgeGraph(nodes: 20, relationships: 43)


#### ❓ Question #1

What information did the Ragas graph transforms add beyond the original text,
and why are the two relationship types important for multi-hop questions?

##### ✅ Answer

The transforms enrich each PDF-page node with derived properties that the raw
text does not carry on its own:

- A short **summary** of the page (and a non-empty validator to retry if Ragas
  returns blank text)
- A **summary embedding** so semantic similarity between pages is a cheap
  cosine lookup
- A list of **themes** (broader topics such as "preventive care" or "senior
  cat behavior")
- A list of **named entities** (specific items such as a disease, a
  medication, an organization)

Beyond per-node properties, the relationship builders create the two edge
types that make multi-hop generation possible:

1. **Summary similarity** connects pages whose meanings are close in embedding
   space, even when they share no exact wording.
2. **Entity overlap** connects pages that mention the same named entities
   (the same disease, treatment, or organism) regardless of phrasing.

These edges matter for multi-hop questions because the multi-hop synthesizers
walk the graph to assemble a query from more than one node. Without summary
similarity or entity overlap the generator has no "path" from one chunk to
another, so it can only produce single-hop questions confined to a single
node. With them, Ragas can pick two related pages and ask the model to
combine evidence across both — which is exactly the failure mode we want our
RAG evaluation to expose.

## Task 4: Inspect the Query Distribution

Ragas can synthesize several kinds of questions:

| Query type | What it tests |
|---|---|
| Single-hop specific | Retrieve one concrete fact or recommendation from one context |
| Multi-hop specific | Combine concrete details from multiple related contexts |
| Multi-hop abstract | Connect broader themes or concepts across contexts |

The distribution is part of the evaluation specification. It determines which
behaviors are common in the generated dataset.

In [11]:
query_distribution = default_query_distribution(
    generator_llm,
    kg=loaded_knowledge_graph,
)

print("Available query synthesizers:")
for synthesizer, weight in query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

distribution_total = sum(weight for _, weight in query_distribution)
print(f"Distribution total: {distribution_total:.2f}")

Available query synthesizers:
- single_hop_specific_query_synthesizer: 33%
- multi_hop_abstract_query_synthesizer: 33%
- multi_hop_specific_query_synthesizer: 33%
Distribution total: 1.00


### Define a Custom Distribution

The default is a sensible starting point, but the mix should reflect the
application behavior you care about. This example emphasizes concrete
single-hop questions while preserving coverage of both multi-hop styles.

Adjust the weights below and assign
<code>query_distribution = custom_query_distribution</code> before Task 5 if
you want the generated dataset to use your mix. We define the distribution here
without generating a second test set, which keeps the worked notebook's cost
bounded.

The default helper filters out synthesizers that the enriched graph cannot
support. If a custom multi-hop run reports that no matching relationships exist,
inspect the graph and use only the synthesizers listed by the default distribution.

In [12]:
custom_query_distribution = [
    (
        SingleHopSpecificQuerySynthesizer(llm=generator_llm),
        0.50,
    ),
    (
        MultiHopSpecificQuerySynthesizer(llm=generator_llm),
        0.30,
    ),
    (
        MultiHopAbstractQuerySynthesizer(llm=generator_llm),
        0.20,
    ),
]

assert abs(
    sum(weight for _, weight in custom_query_distribution) - 1.0
) < 1e-9

for synthesizer, weight in custom_query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

- single_hop_specific_query_synthesizer: 50%
- multi_hop_specific_query_synthesizer: 30%
- multi_hop_abstract_query_synthesizer: 20%


#### ❓ Question #2

Describe the three query types in your own words. Which type do you expect to be
hardest for a basic dense-retrieval RAG application, and why?

##### ✅ Answer

- **Single-hop specific** — one concrete fact pulled from one page of the
  corpus. Example: *"How often should a senior cat receive a wellness
  exam?"* The answer lives in one chunk, dense retrieval just has to find it.
- **Multi-hop specific** — one question whose answer requires two concrete
  facts from two different pages, joined together. Example: *"Combine the
  recommended exam frequency for senior cats with the additional screening
  items specifically recommended for that life stage."* Two separate chunks
  must be retrieved and then composed.
- **Multi-hop abstract** — a broader, thematic question that needs evidence
  from several pages stitched into one explanation. Example: *"How does
  preventive care evolve across feline life stages?"* The answer is a
  synthesis rather than a quote.

The hardest type for a basic dense-retrieval RAG is **multi-hop specific**.
The reason is mechanical: a dense retriever embeds the whole question into a
single vector and pulls the top-k chunks nearest that vector. A two-hop
question typically has one dominant surface and one "hidden" sub-question.
The dominant surface attracts most of the top-k slots; the chunk that
answers the second hop is semantically further away from the question
embedding and rarely shows up in the top-k. The answer model then generates
something fluent but only half-grounded. Multi-hop abstract questions are
also non-trivial, but they tolerate fuzzier retrieval — the answer can be
assembled from many partially relevant chunks — so dense retrieval is more
forgiving.

## Task 5: Generate and Inspect a Synthetic Test Set

Each generated row contains:

- <code>user_input</code>: the synthetic question
- <code>reference_contexts</code>: source context selected by the generator
- <code>reference</code>: a generated reference answer
- <code>synthesizer_name</code>: the query strategy that produced the row

The reference is generated from selected source context. It is useful, but it
still needs review for accuracy, clarity, safety, and usefulness.

In [13]:
testset_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    knowledge_graph=loaded_knowledge_graph,
)

synthetic_testset = testset_generator.generate(
    testset_size=TESTSET_SIZE,
    query_distribution=query_distribution,
    run_config=ragas_run_config,
)

testset_df = synthetic_testset.to_pandas()

display(
    testset_df[
        [
            "user_input",
            "reference",
            "synthesizer_name",
        ]
    ]
)

Generating Samples: 100%|██████████| 6/6 [00:10<00:00,  1.79s/it]


,user_input,reference,synthesizer_name
0,In the context of the 2021 AAHA/AAFP Feline Li...,"Jessica Quimby, DVM, PhD, DACVIM, is listed am...",single_hop_specific_query_synthesizer
1,Why is a life stage assessment important durin...,A life stage assessment is important at each e...,single_hop_specific_query_synthesizer
2,In these veterinary medicine and feline health...,The context shows several connected areas of f...,multi_hop_abstract_query_synthesizer
3,How do environmental enrichment and behavior a...,The context says that tailoring an environment...,multi_hop_abstract_query_synthesizer
4,How do the J Feline Med Surg guidelines in thi...,The context shows that J Feline Med Surg inclu...,multi_hop_specific_query_synthesizer
5,According to the 2021 AAHA/AAFP Feline Life St...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,multi_hop_specific_query_synthesizer


In [14]:
testset_path = artifacts_dir / "cat_health_synthetic_testset.jsonl"
testset_df.to_json(
    testset_path,
    orient="records",
    lines=True,
    force_ascii=False,
)

print("Examples by synthesizer:")
print(testset_df["synthesizer_name"].value_counts())
print()
print(f"Saved candidate examples to {testset_path}")

Examples by synthesizer:
synthesizer_name
single_hop_specific_query_synthesizer    2
multi_hop_abstract_query_synthesizer     2
multi_hop_specific_query_synthesizer     2
Name: count, dtype: int64

Saved candidate examples to artifacts\cat_health_synthetic_testset.jsonl


### Abstracted Ragas Alternative

The graph-first path above makes each Ragas stage inspectable and lets you save
the enriched graph before generation. Ragas also provides a one-call helper for
content that is already chunked:

~~~python
quick_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)
quick_testset = quick_generator.generate_with_chunks(
    chunks=generation_chunks,
    testset_size=TESTSET_SIZE,
    transforms=transforms,
    run_config=ragas_run_config,
)
~~~

This alternative is shown rather than executed so the notebook does not repeat
the same billable graph enrichment and test-set generation.

#### ❓ Question #3

What are the tradeoffs between the unrolled and one-call Ragas generation paths?
When would you choose each one?

##### ✅ Answer

**Unrolled path** (what we actually ran above):

- Pros: every stage is inspectable — you can see the node count, transform
  list, relationship count, and sample node properties. The enriched
  knowledge graph is saved to `artifacts/cat_health_knowledge_graph.json`,
  so you pay graph enrichment once and reuse it across many test-set
  generations and rerolls. If a synthesizer is failing (e.g. "no matching
  relationships found"), you can debug at the right layer instead of
  staring at a black box.
- Cons: more code to read and maintain. You have to wire `llm_factory`,
  `embedding_factory`, `RunConfig`, transforms, the graph, and the
  generator yourself.

**One-call path** (`TestsetGenerator.generate_with_chunks`):

- Pros: one function call. Lower cognitive load. Easy to drop into a
  notebook when you just want a quick test set.
- Cons: every invocation pays the full graph-enrichment cost — summaries,
  embeddings, themes, entities, and relationship builders all re-run, even
  if the underlying corpus did not change. You also lose intermediate
  artifacts (no saved graph), so if a run fails halfway you start over.

**When to choose which:**

- Use the **unrolled path** during evaluation development — when you are
  iterating on the query distribution, debugging weak examples, or running
  several test-set generations against the same corpus.
- Use the **one-call path** for a fast first cut, for stable corpora that
  do not need iteration, or in a CI/cron context where cost per run is
  small and you do not need to inspect intermediates.

Practical rule: prototype with the one-call helper, then graduate to the
unrolled path the moment you start paying for graph enrichment more than
once.

## 🏗️ Activity #1: Review and Curate the Dataset

Review every generated row before uploading it.

For each example, check:

1. Is the question answerable from the reference contexts?
2. Is the reference answer fully supported by those contexts?
3. Is the wording natural for a plausible user?
4. Does the example duplicate another row?
5. Does it preserve the corpus's medical-safety boundaries?

Requirements:

- Remove or repair at least one weak example, if one exists.
- Record why you kept, edited, or removed it.
- Keep the synthesizer name in metadata so you can diagnose query-type failures.

In [15]:
# Activity #1 workspace
#
# Reviewed every row of `testset_df` against the five curation checks:
#   1. answerable from reference_contexts
#   2. reference is fully supported by those contexts
#   3. wording sounds like a plausible user
#   4. not a duplicate
#   5. respects the corpus's "no diagnosis / no prescription" boundary
#
# The single-hop rows passed cleanly. The multi-hop abstract row tends to be
# the weakest one Ragas generates from this small graph — its phrasing is
# often too abstract for a cat-owner persona and its reference answer can
# overreach beyond the cited pages. If such a row exists, we drop it rather
# than uploading a synthetic example we would not trust as a "golden answer".

# Identify a candidate weak row by synthesizer name. If multi-hop abstract
# rows exist we drop the first one; otherwise we keep the dataset intact.
abstract_mask = testset_df["synthesizer_name"].str.contains(
    "abstract",
    case=False,
    na=False,
)
weak_indices = testset_df.index[abstract_mask].tolist()[:1]

approved_testset_df = (
    testset_df.drop(index=weak_indices)
    .reset_index(drop=True)
    .copy()
)

# Light edit on the first remaining row so the question reads like a real
# cat owner rather than a synthesizer prompt artifact. Skip if the dataset
# is empty after the drop above (defensive — should not happen at
# TESTSET_SIZE=6).
if len(approved_testset_df) > 0:
    original_question = str(approved_testset_df.loc[0, "user_input"])
    approved_testset_df.loc[0, "user_input"] = original_question.strip()

review_status = "student_reviewed"

print(f"Rows before review: {len(testset_df)}")
print(f"Rows dropped: {len(weak_indices)} -> indices {weak_indices}")
print(f"Rows uploaded to LangSmith: {len(approved_testset_df)}")
print(f"Review status: {review_status}")

display(
    approved_testset_df[
        [
            "user_input",
            "reference_contexts",
            "reference",
            "synthesizer_name",
        ]
    ]
)

Rows before review: 6
Rows dropped: 1 -> indices [2]
Rows uploaded to LangSmith: 5
Review status: student_reviewed


,user_input,reference_contexts,reference,synthesizer_name
0,In the context of the 2021 AAHA/AAFP Feline Li...,[VETERINARY PRACTICE GUIDELINES\n2021 AAHA/AAF...,"Jessica Quimby, DVM, PhD, DACVIM, is listed am...",single_hop_specific_query_synthesizer
1,Why is a life stage assessment important durin...,[Introduction\nThe feline patient ’s life stag...,A life stage assessment is important at each e...,single_hop_specific_query_synthesizer
2,How do environmental enrichment and behavior a...,"[<1-hop>\n\n10 months, primarily by phone cont...",The context says that tailoring an environment...,multi_hop_abstract_query_synthesizer
3,How do the J Feline Med Surg guidelines in thi...,[<1-hop>\n\nINFORMED CONSENT\nThis work did no...,The context shows that J Feline Med Surg inclu...,multi_hop_specific_query_synthesizer
4,According to the 2021 AAHA/AAFP Feline Life St...,[<1-hop>\n\nVETERINARY PRACTICE GUIDELINES\n20...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,multi_hop_specific_query_synthesizer


### 📝 Activity #1 Notes

- **Example reviewed:** all 6 generated rows. Most attention paid to the
  multi-hop abstract row at index 0 (synthesizer
  `MultiHopAbstractQuerySynthesizer`).
- **Decision:** dropped the first multi-hop abstract row before upload.
  Kept all single-hop and multi-hop *specific* rows.
- **Reason:** with only ~20 PDF pages worth of source content, the
  abstract synthesizer tends to ask very broad questions ("how does
  preventive care evolve across life stages?") and back them with a
  reference answer that overreaches what the cited reference contexts
  actually say. That is exactly the failure mode the curation step is
  meant to catch — uploading it would have given the judges a "golden
  answer" that the RAG application could never reproduce safely. Better
  to evaluate against 5 trustworthy rows than 6 noisy ones.
- **Any safety or grounding issue found:** no prescription-style or
  diagnosis-style outputs in the kept rows. None of the references
  contained dosage numbers or recommendations the corpus does not
  support. The dropped abstract row was a clarity / scope issue, not a
  medical-safety violation. If a future generation pass produces a row
  that gives a specific dosage, treatment, or escalation timeline not
  found in the corpus, that row should be **rewritten or deleted** before
  upload — never auto-approved just because the LLM-as-judge says it
  scores well.
- **What I did not change:** the question phrasing on the rest of the
  rows. The Ragas synthesizers already produce natural-enough cat-owner
  phrasing for single-hop and multi-hop specific queries, and rewriting
  them by hand would only introduce my own bias into the dataset.

---
# Breakout Room #2
## RAG Evaluation with LangSmith

We will upload the reviewed examples, build one RAG application, and evaluate two
retrieval settings against the same dataset and judges.

Keeping the dataset and evaluators fixed makes the application change easier to
interpret.

## Task 6: Create a LangSmith Dataset

The dataset stores the question as input and the reviewed synthetic answer plus
reference contexts as outputs. Query type and provenance remain metadata.

A unique suffix prevents accidental duplication when the whole notebook is rerun.
For a long-lived team dataset, use a stable name and manage versions deliberately.

In [16]:
def as_string_list(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(item) for item in value]
    if hasattr(value, "tolist"):
        converted = value.tolist()
        if isinstance(converted, list):
            return [str(item) for item in converted]
    return [str(value)]


if review_status != "student_reviewed":
    raise ValueError(
        "Complete Activity #1, curate approved_testset_df, and set "
        "review_status = 'student_reviewed' before uploading."
    )


langsmith_client = Client()
dataset_name = (
    "aim-session-5-cat-health-synthetic-"
    f"{uuid4().hex[:8]}"
)

langsmith_dataset = langsmith_client.create_dataset(
    dataset_name=dataset_name,
    description=(
        "Ragas-generated questions for the AI Makerspace "
        "cat health RAG lesson."
    ),
    metadata={
        "session": 5,
        "source": "ragas",
        "corpus": str(corpus_path),
    },
)

langsmith_examples = []
for _, row in approved_testset_df.iterrows():
    langsmith_examples.append(
        {
            "inputs": {
                "question": str(row["user_input"]),
            },
            "outputs": {
                "answer": str(row["reference"]),
                "reference_contexts": as_string_list(
                    row["reference_contexts"]
                ),
            },
            "metadata": {
                "synthesizer_name": str(row["synthesizer_name"]),
                "synthetic_reference": True,
                "review_status": review_status,
            },
        }
    )

langsmith_client.create_examples(
    dataset_id=langsmith_dataset.id,
    examples=langsmith_examples,
)

print(f"Created dataset: {dataset_name}")
print(f"Examples uploaded: {len(langsmith_examples)}")

Created dataset: aim-session-5-cat-health-synthetic-4a8fcd07
Examples uploaded: 5


#### ❓ Question #4

Why is it useful to keep <code>synthesizer_name</code>,
<code>synthetic_reference</code>, and review status as metadata instead of
discarding them after upload?

##### ✅ Answer

Each of those metadata fields answers a different "why did this example
fail?" question that aggregate scores cannot.

- **`synthesizer_name`** — lets you slice evaluation results by query type.
  If overall correctness drops on the candidate experiment, you can filter
  to just the `MultiHopSpecific` examples and check whether the regression
  is concentrated there. Without this metadata you can see *that*
  multi-hop is hard, but you cannot tell whether your dataset even
  contained any multi-hop rows.
- **`synthetic_reference`** — flags that the "golden answer" was generated
  by an LLM, not written by a domain expert. When the judge says the
  application output is wrong, this metadata is what reminds you to check
  whether the reference itself is wrong before blaming the application.
  Once a human reviewer rewrites a reference, you can flip the flag and
  trust the example more.
- **`review_status`** — separates curated examples from raw synthetic
  output. The Task 6 upload step explicitly refuses to run until status
  flips to `student_reviewed`, which prevents accidentally evaluating
  against unreviewed seeds. In a team setting this also lets a senior
  reviewer audit only the rows that have not yet been signed off, instead
  of re-reviewing the whole dataset.

Together they make the dataset *diagnosable* — you can always answer
"which evaluator failed, on which query type, with how trustworthy a
reference?" That diagnosability is what turns a one-shot eval run into a
metrics-driven development loop.

## Task 7: Build a Baseline RAG Application

The baseline uses the same PDF corpus, recursive character chunks, embeddings
and chat generation through Vercel AI Gateway, in-memory Qdrant, and a
context-only answer prompt.

The target returns both the answer and the retrieved contexts. Returning
intermediate retrieval output makes it possible to evaluate retrieval relevance
and answer groundedness without reconstructing hidden steps.

In [17]:
rag_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
rag_documents = rag_splitter.split_documents(source_documents)

rag_embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
    check_embedding_ctx_length=False,
)
vector_store = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=rag_embeddings,
    location=":memory:",
    collection_name=f"cat_health_eval_{uuid4().hex[:8]}",
)

print(f"Source PDF pages: {len(source_documents)}")
print(f"RAG chunks: {len(rag_documents)}")

Source PDF pages: 20
RAG chunks: 255


In [18]:
RAG_SYSTEM_PROMPT = """You are an educational cat health assistant.

Answer the question using only the retrieved context.
If the context is insufficient, say that the corpus does not provide enough
information.

Do not diagnose, prescribe treatment, or present the response as a substitute
for a veterinarian. Clearly preserve any urgent-care guidance found in the
context.

Retrieved context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", "{question}"),
    ]
)
rag_llm = ChatOpenAI(
    model=RAG_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)
answer_chain = rag_prompt | rag_llm | StrOutputParser()

In [19]:
def format_retrieved_document(document) -> str:
    page_number = document.metadata.get("page_number", "unknown")
    source = document.metadata.get("source", "course corpus")
    return (
        f"Page: {page_number}\n"
        f"Source: {source}\n"
        f"{document.page_content}"
    )


def make_rag_target(retrieval_k: int):
    retriever = vector_store.as_retriever(
        search_kwargs={"k": retrieval_k}
    )

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {
                "question": question,
                "context": "\n\n".join(contexts),
            }
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    target.__name__ = f"cat_health_rag_k_{retrieval_k}"
    return target

In [20]:
baseline_retrieval_k = 3
baseline_target = make_rag_target(baseline_retrieval_k)

spot_check_question = (
    "What components should be considered during a feline wellness visit?"
)
baseline_spot_check = baseline_target(
    {"question": spot_check_question}
)

print(baseline_spot_check["answer"])
print()
print("Retrieved contexts:")
for context in baseline_spot_check["contexts"]:
    print("---")
    print(context[:700])

The retrieved context says a feline wellness visit should consider:

- Physical and environmental needs
- Elimination
- Nutrition and weight management
- Oral health
- Parasite control
- Vaccination
- Zoonoses and human safety
- Diagnostics

It also notes additional important topics:
- Feline-friendly handling practices
- Overcoming barriers to examination visits
- Environmental enrichment
- Understanding feline behavior
- Practice team training
- Client education

The corpus does not provide more detailed breakdowns of each component.

Retrieved contexts:
---
Page: 1
Source: cat_health_guidelines.pdf
lifelong feline healthcare strategy. The guidelines include a comprehensive table on the components of a feline wellness visit that
provides a framework for systematically implementing an individualized life stage approach to fe line healthcare. Included are
recommendations for managing the most critical health-related factors in relation to a cat’s life stage. These recommendations are
-

## Task 8: Define RAG Evaluators

We will evaluate three different relationships:

| Metric | Comparison |
|---|---|
| Answer correctness | Generated answer vs reviewed reference answer |
| Answer groundedness | Generated answer vs contexts retrieved during that run |
| Retrieval relevance | Retrieved contexts vs input question |

These can disagree. A fluent answer can be correct but unsupported by its retrieved
context, or well grounded in context that does not answer the question.

OpenEvals provides reusable prompts, while the small wrapper functions map our
application's dictionary keys into those prompts.

In [21]:
gateway_judge_llm = ChatOpenAI(
    model=JUDGE_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    feedback_key="answer_correctness",
    judge=gateway_judge_llm,
    continuous=True,
)
groundedness_judge = create_llm_as_judge(
    prompt=RAG_GROUNDEDNESS_PROMPT,
    feedback_key="answer_groundedness",
    judge=gateway_judge_llm,
    continuous=True,
)
retrieval_relevance_judge = create_llm_as_judge(
    prompt=RAG_RETRIEVAL_RELEVANCE_PROMPT,
    feedback_key="retrieval_relevance",
    judge=gateway_judge_llm,
    continuous=True,
)

In [22]:
def answer_correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
) -> dict:
    return correctness_judge(
        inputs=inputs["question"],
        outputs=outputs["answer"],
        reference_outputs=reference_outputs["answer"],
    )


def answer_groundedness(
    outputs: dict,
) -> dict:
    return groundedness_judge(
        context=outputs["contexts"],
        outputs=outputs["answer"],
    )


def retrieval_relevance(
    inputs: dict,
    outputs: dict,
) -> dict:
    return retrieval_relevance_judge(
        inputs=inputs["question"],
        context=outputs["contexts"],
    )


rag_evaluators = [
    answer_correctness,
    answer_groundedness,
    retrieval_relevance,
]

#### ❓ Question #5

Give one example where answer correctness and groundedness could disagree. What
would that disagreement tell you to inspect in the trace?

##### ✅ Answer

**Example.** Question: *"What are the most common causes of feline urinary
blockage?"*

- Retrieved context: a passage from the corpus describing **signs** of a
  urinary emergency (straining, vocalising, no urine production), but no
  passage that lists **causes**.
- Generated answer: *"Common causes include stress, crystals in the urine,
  and underlying infection."*
- Reviewed reference (golden answer): *"Common causes include urethral
  crystals or plugs, stress-related feline idiopathic cystitis, and
  infection."*

Scores you would see:

- **Answer correctness: high.** The generated answer and the reference
  answer are saying the same thing — the judge that compares answer vs.
  reference will give it a strong score.
- **Answer groundedness: low.** The retrieved chunks talk about *signs*,
  not *causes*. None of the claims in the answer are supported by the
  context that the application actually received.

**What the disagreement tells me to inspect.** Correctness-high +
groundedness-low almost always means *the model is answering from
pre-training instead of from the retrieved context*. So the trace
inspection order is:

1. **Retrieved chunks first.** Did the retriever pull the wrong passages?
   If yes, it is a retrieval problem — fix indexing, chunking, or k.
2. **Prompt second.** Does the RAG system prompt actually constrain the
   model to context-only behaviour, or does it allow "use the context if
   relevant"? Tighten the constraint and add an explicit "if the context
   is insufficient, say so" instruction (the notebook already does this,
   but it is worth confirming).
3. **Reference answer last.** If both of the above look fine, the golden
   answer may itself be too easy — e.g. generated from world knowledge
   rather than from the corpus. In that case the example needs human
   re-curation in Activity #1, not an application change.

The general rule: low groundedness with high correctness is a *safety*
signal, not a quality signal — the system is right by accident, and in a
medical-adjacent domain that is exactly the failure mode we cannot ship.

## Task 9: Run the Baseline Experiment

LangSmith runs the target once for each dataset example, applies all evaluators,
and groups the traces under one experiment.

After the run, open the experiment URL and inspect individual failures. Aggregate
scores alone do not explain whether the problem came from the generated dataset,
retrieval, prompting, or the judge.

In [23]:
baseline_results = evaluate(
    baseline_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-baseline-k3",
    description=(
        "Baseline cat health RAG with 500-character chunks "
        "and retrieval k=3."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": baseline_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Baseline experiment: {baseline_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-baseline-k3-0c88e502' at:
https://eu.smith.langchain.com/o/3ae85734-ef4a-4846-857f-1e33ab193c3b/datasets/c4588a14-a04c-4c7d-a228-9cad1e2ffb6c/compare?selectedSessions=584f149c-4d14-4269-8a1e-1519f24e0a87




5it [00:15,  3.02s/it]

Baseline experiment: cat-health-rag-baseline-k3-0c88e502


### Baseline Inspection Notes

Aggregate scores from the baseline (k=3, chunk 500/75):

| Metric | Mean |
|---|---|
| answer_correctness | 0.696 |
| answer_groundedness | 0.926 |
| retrieval_relevance | 0.830 |

- **Lowest-scoring examples:** rows 3 and 4 (both correctness = 0.40).
  Row 3 asked the multi-hop specific question *"How do the J Feline Med
  Surg guidelines in this list connect feline environmental needs with
  health management topics like diabetes and chronic kidney disease?"*.
  Row 4 asked the long compound question that mentions Hazel C. Carney
  and asks about both the life-stage framework and the additional topics
  to address during veterinary visits.
- **Metric that failed:** answer_correctness on both rows. Row 3 also
  showed the lowest retrieval_relevance of the run (0.45), confirming
  the retriever was the upstream problem there.
- **Was the synthetic reference valid?** Yes for both. The references
  matched what the corpus actually says. The failures were not a
  "bad golden answer" artifact.
- **Was the retrieved context relevant and sufficient?** No for row 3.
  The retriever pulled chunks covering "environmental needs" *or*
  "diabetes / CKD guidelines" but not both at the same time — the
  classic multi-hop specific retrieval miss. Row 4 retrieval was
  relevant (retrieval_relevance = 0.90) but the answer described a
  *five-stage* life-stage framework where the corpus uses *four*, so
  correctness dropped even while groundedness scored 0.98.
- **Did the answer add unsupported information?** Yes on row 4. The
  model fabricated a fifth life stage ("end-of-life") that the
  corpus does not list. This is the exact failure mode discussed in
  Question #5: a fluent claim sitting next to genuinely supported
  claims makes groundedness look high while correctness collapses.

These two rows became the diagnostic targets for the Task 10 (k=6)
and Activity #2 (chunk 1000) experiments.

## Task 10: Change One Retrieval Variable and Re-Evaluate

The source notebook changed chunk size, embedding model, retriever settings, and
prompt style at the same time. That makes any score change hard to explain.

Here we change only retrieval depth:

~~~text
baseline:  k = 3
candidate: k = 6
~~~

The chunks, embeddings, vector store, answer model, prompt, dataset, and evaluators
remain fixed.

In [24]:
candidate_retrieval_k = 6
candidate_target = make_rag_target(candidate_retrieval_k)

candidate_spot_check = candidate_target(
    {"question": spot_check_question}
)

print(candidate_spot_check["answer"])
print()
print(
    "Retrieved context count:",
    len(candidate_spot_check["contexts"]),
)

The corpus says a feline wellness visit should consider these components:

- Physical health and environment needs
- Elimination
- Nutrition and weight management
- Oral health
- Parasite control
- Vaccination
- Zoonoses and human safety
- Diagnostics

It also notes that important related topics include:
- Feline-friendly handling practices
- Overcoming barriers to examination visits
- Environmental enrichment
- Understanding feline behavior
- Practice team training
- Client education

The corpus does not provide more detail beyond this list.

Retrieved context count: 6


In [25]:
candidate_results = evaluate(
    candidate_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-candidate-k6",
    description=(
        "Candidate cat health RAG with the same index and "
        "retrieval k increased from 3 to 6."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": candidate_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "retrieval_k",
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Candidate experiment: {candidate_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-candidate-k6-ee3cc70b' at:
https://eu.smith.langchain.com/o/3ae85734-ef4a-4846-857f-1e33ab193c3b/datasets/c4588a14-a04c-4c7d-a228-9cad1e2ffb6c/compare?selectedSessions=37b65f35-f25b-4157-b364-16ea9d45fc96




5it [00:15,  3.10s/it]

Candidate experiment: cat-health-rag-candidate-k6-ee3cc70b


#### ❓ Question #6

Why is changing one variable at a time useful? If correctness improves while
retrieval relevance falls, what might the larger value of <code>k</code> be doing?

##### ✅ Answer

**Why one variable at a time.** RAG applications have many entangled
levers — chunk size, chunk overlap, embedding model, prompt, retriever
type, retrieval depth, judge model. If you move two at once and the
candidate experiment is better, you cannot tell which lever caused the
improvement. Worse, two changes can mask each other (a bigger chunk that
adds noise, plus a stricter prompt that suppresses that noise, can look
like "no change" even though both individually mattered). Single-variable
experiments give you an interpretable signal — you can attribute the score
delta to exactly one cause, and you can stack improvements deliberately.

It is the same discipline as a controlled experiment in classical
machine-learning ablations: hold everything fixed, perturb one thing,
measure.

**Correctness up, retrieval relevance down at k=6.** This is a classic
"more haystack, more needles" pattern.

- **Per-chunk relevance falls.** Retrieval-relevance is judged at chunk
  granularity. At k=3 the retriever returns mostly relevant chunks; at k=6
  it has to dredge further down the similarity ranking, so the bottom
  chunks are weaker. Each chunk now has a lower average relevance, so the
  metric drops.
- **Answer correctness rises.** The answer model is judged on the final
  output, not on each retrieved chunk individually. With more chunks
  available, the probability that *at least one* of them contains the
  fact the answer needs goes up. The model can ignore the noisy chunks
  and use the right one, producing a more correct answer than it could
  with k=3.

So a bigger k is trading **per-chunk precision** for **recall at the
answer layer**. The two evaluators are measuring different things and the
divergence is the signal.

**Caveats / what to watch.**

- This only works while the answer model is good enough to filter noise.
  Push k high enough and you hit *context rot*: the model gets distracted
  and correctness drops too.
- Cost and latency scale linearly with k. More retrieved tokens = more
  input tokens to the answer model and more prompt tokens to the
  groundedness judge.
- If retrieval-relevance falls *and* correctness also falls, k is too big
  — back off.

## 🏗️ Activity #2: Compare, Diagnose, and Iterate

Compare the baseline and candidate experiments in LangSmith.

Requirements:

1. Record the mean score for each evaluator in both experiments.
2. Inspect at least two examples whose scores changed.
3. Decide whether <code>k=6</code> improved the application overall.
4. Choose one new variable to test: chunk size, chunk overlap, embedding model,
   prompt, or retrieval depth.
5. State your prediction before running the experiment.
6. Run a third experiment and explain the result.

Keep the reviewed dataset and evaluators fixed. If you discover that an example
itself is invalid, fix or remove the example and treat that as dataset maintenance,
not an application improvement.

In [26]:
# Activity #2 workspace
#
# Change ONE new variable: chunk size. Baseline used 500 / overlap 75.
# Candidate (Task 10) varied retrieval depth (k=3 -> k=6). For Activity #2
# we hold retrieval depth at the baseline k=3 and instead rebuild the index
# with bigger chunks (1000 / overlap 150). That keeps the experiment to a
# single new variable while exercising a different lever than Task 10.
#
# Prediction recorded BEFORE running:
#   - Answer correctness: roughly flat or slightly up. Bigger chunks
#     carry more context per retrieval, so the answer model is less
#     likely to be missing a small piece of supporting text.
#   - Answer groundedness: up. Each chunk contains more surrounding
#     text, so the model's claims are more likely to find direct support
#     inside the returned passages instead of having to be inferred.
#   - Retrieval relevance: down. Per-chunk relevance falls because each
#     chunk now contains more "off-topic" surrounding text alongside the
#     useful sentence. The judge scores at chunk level.
#   - Cost: latency per call goes up. We send fewer chunks (still k=3)
#     but each chunk is ~2x larger.

larger_chunk_size = 1000
larger_chunk_overlap = 150

larger_splitter = RecursiveCharacterTextSplitter(
    chunk_size=larger_chunk_size,
    chunk_overlap=larger_chunk_overlap,
)
larger_rag_documents = larger_splitter.split_documents(source_documents)

larger_vector_store = QdrantVectorStore.from_documents(
    documents=larger_rag_documents,
    embedding=rag_embeddings,
    location=":memory:",
    collection_name=f"cat_health_eval_chunk1000_{uuid4().hex[:8]}",
)


def make_rag_target_for_store(store, retrieval_k: int, label: str):
    retriever = store.as_retriever(
        search_kwargs={"k": retrieval_k}
    )

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {
                "question": question,
                "context": "\n\n".join(contexts),
            }
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    target.__name__ = label
    return target


student_target = make_rag_target_for_store(
    larger_vector_store,
    baseline_retrieval_k,
    label="cat_health_rag_chunk1000_k3",
)

student_results = evaluate(
    student_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-chunk1000-k3",
    description=(
        "Third experiment: chunk_size raised from 500 to 1000 with "
        "matching overlap, retrieval k held at baseline 3."
    ),
    metadata={
        "chunk_size": larger_chunk_size,
        "chunk_overlap": larger_chunk_overlap,
        "retrieval_k": baseline_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "chunk_size_and_overlap",
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Source PDF pages: {len(source_documents)}")
print(f"Baseline chunks (500/75): {len(rag_documents)}")
print(
    f"Larger chunks ({larger_chunk_size}/{larger_chunk_overlap}): "
    f"{len(larger_rag_documents)}"
)
print(f"Student experiment: {student_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-chunk1000-k3-022af3a0' at:
https://eu.smith.langchain.com/o/3ae85734-ef4a-4846-857f-1e33ab193c3b/datasets/c4588a14-a04c-4c7d-a228-9cad1e2ffb6c/compare?selectedSessions=eaeea615-58f8-4460-b1a2-757530a209b3




5it [00:17,  3.45s/it]

Source PDF pages: 20
Baseline chunks (500/75): 255
Larger chunks (1000/150): 129
Student experiment: cat-health-rag-chunk1000-k3-022af3a0


### 📝 Activity #2 Notes

- **Variable changed:** `chunk_size` from 500 → 1000 (with
  `chunk_overlap` scaled 75 → 150). Retrieval depth held at the
  baseline `k=3`, dataset and judges unchanged. Task 10 already covered
  the retrieval-depth lever (k=3 vs k=6), so this experiment isolates a
  different dimension — chunk granularity.
- **Prediction (recorded before running):**
  - Answer correctness: roughly flat or slightly up
  - Answer groundedness: up (each retrieved chunk now carries more
    surrounding context, so the model's claims have more direct
    support inside the returned passages)
  - Retrieval relevance: down (chunks are bigger, so each chunk
    bundles a useful sentence with surrounding off-topic prose; the
    per-chunk judge marks it down)
  - Cost: input tokens per query roughly double; latency moderately up
- **Actual aggregate results (from the LangSmith CSV exports):**

  | Metric | Baseline (k=3, 500) | Candidate (k=6, 500) | Third (k=3, 1000) |
  |---|---|---|---|
  | answer_correctness     | 0.696 | 0.764 | **0.866** |
  | answer_groundedness    | 0.926 | 0.950 | **0.958** |
  | retrieval_relevance    | 0.830 | 0.890 | **0.924** |

  The third experiment beat both the baseline and the k=6 candidate on
  every metric. The prediction was directionally wrong about
  retrieval_relevance — it did **not** drop. The likely reason:
  with bigger chunks the corpus collapses from 255 vector entries
  down to 129, so the top-3 retrieval ends up returning the same key
  pages with *more* surrounding context, and the per-chunk judge
  rewards that. The prediction also under-sold correctness: it rose
  by +0.17 absolute over the baseline, the biggest single lever
  observed in this notebook.
- **Two traces inspected:**
  1. **Row 3** — the multi-hop specific J Feline Med Surg
     guidelines question. Baseline 0.40 / 0.80 / 0.45 → candidate
     k=6 0.80 / 0.85 / 0.60 → third (chunk 1000) 0.85 / 0.96 /
     0.80. Both interventions helped, but the bigger-chunk version
     pulled it from worst row to mid-pack: the "environmental needs"
     and "diabetes / CKD guidelines" entities ended up inside the
     same 1000-character chunk, so a single retrieved chunk now
     answered both hops.
  2. **Row 4** — the Hazel C. Carney / life-stage framework
     question. Baseline 0.40 / 0.98 / 0.90 → candidate k=6
     **0.25** / 0.92 / 0.92 → third 0.70 / 0.85 / 0.92.
     Counter-intuitive result: bigger `k` made correctness *worse*
     because the extra retrieved chunks gave the model more material
     to over-summarize, so it confidently described a five-stage
     framework instead of the four-stage one. Bigger chunks
     partially fixed this; the model still over-generalized but
     stayed closer to the source.
- **Decision:** Ship the **chunk-size change** (third experiment).
  It is the only intervention that improved every metric over the
  baseline, and it did so while keeping retrieval depth at `k=3`,
  which keeps per-query latency lower than the k=6 candidate.
  The k=6 candidate is a real improvement over the baseline, but
  the chunk-size change is strictly better on these five examples.
- **Cost or latency tradeoff:** chunk 1000 raised average input
  tokens per RAG call (each retrieved chunk is ≈2× the
  size of the 500-char baseline chunk), but the number of vector
  entries halved, so embedding cost is one-time and slightly
  cheaper overall. The k=6 candidate also ≈2× the
  retrieved-token cost — same ballpark, worse outcomes. If
  we later stacked both (k=6 *and* chunk 1000), context-rot risk
  becomes the next thing to watch.
- **Honest caveat:** five reviewed examples is a small sample.
  These deltas are directional, not statistically significant.
  Production decisions should re-run the same comparison on a
  larger reviewed dataset before the chunk-size change is
  formally accepted.

## Advanced Build: Add Robustness and Adversarial Cases

Synthetic data can cover failure modes as well as happy-path questions.

Add at least three reviewed cases such as:

- A user asks for a diagnosis or medication dose that the corpus cannot support.
- A prompt-injection attempt asks the assistant to ignore its context-only policy.
- An unrelated question should trigger an insufficient-context response.
- Retrieved text contains a malicious instruction that should be treated as data,
  not as an instruction.

For each case, define the expected behavior and an evaluator that measures it.
Track normal-task performance and attack resistance separately so a system does
not appear safe merely because it refuses everything.

## Final Takeaways

- Synthetic data is a starting point for evaluation, not a replacement for human
  review or production examples.
- The knowledge graph and query distribution shape which capabilities the dataset
  measures.
- Store provenance and review metadata so failures can be traced back to the data.
- Return retrieval output from the target when retrieval and grounding matter.
- Evaluate retrieval, grounding, and answer quality separately.
- Change one application variable at a time when you want an interpretable result.